# Encyclopedia header — `dynamic_routh_p2_two_quadrics` (03_dynamics_and_routh)

This is the **SolveIt_Notebooks_for_rust encyclopedia copy** of this notebook. Everything below is exact and copy-paste runnable in PowerShell from the **repository root** (the cloned `rustSolveIt_Win11_SUNDIALS_7_8_0` folder — it contains both `posim/` and `SolveIt_Notebooks_for_rust/`).

## 1. Execute this notebook

One-time build of the pure-Rust engine behind it:
```powershell
cargo build --release -p posim
cd dataset_tools; cargo build --release; cd ..   # the SQLite dataset sidecar (solveit_db.exe)
```
Batch execution of this copy (writes real outputs back into it):
```powershell
python SolveIt_Notebooks_for_rust/_tools/run_copy.py SolveIt_Notebooks_for_rust/03_dynamics_and_routh/dynamic_routh_p2_two_quadrics/dynamic_routh_p2_two_quadrics.ipynb
```
Interactive execution (Shift+Enter through every cell, Python 3 (ipykernel) kernel):
```powershell
python -m pip install --user jupyterlab   # once
cd SolveIt_Notebooks_for_rust/03_dynamics_and_routh/dynamic_routh_p2_two_quadrics; jupyter lab dynamic_routh_p2_two_quadrics.ipynb
```
(The notebook finds the simulator itself — `posim.exe` under `target/release/`; if needed set `$env:POSIM_BIN = "$PWD/target/release/posim.exe"`.)

## 2. Display the browser GUI

This notebook's GUI is posim's own live **scene window**. Replay the paired scene and open it:
```powershell
python SolveIt_Notebooks_for_rust/03_dynamics_and_routh/dynamic_routh_p2_two_quadrics/player_dynamic_routh_p2_two_quadrics.py gui
```
Manually, in the interactive REPL (`target/release/posim.exe`): type the commands of `dynamic_notebooks/routh_p2_two_quadrics.posim`, then `SCENE CREATE 8900` and open http://127.0.0.1:8900/ in a browser.

## 3. Movies

No pre-recorded movie pairs with this notebook. To watch it as a movie: open the live scene window (section 2) — the scene plays in real time — and screen-record it (Windows: Win+Alt+R starts the Xbox Game Bar recorder, or use the Snipping Tool's Record mode). The 13 `video_*` notebooks in `05_mechanism_videos/` are the family with recorded movie pages.

## 4. Database / data access

Executing this notebook records the run into a **SQLite dataset**, `datasets/nb_dynamic_routh_p2_two_quadrics.db` (schema v1: `meta`, `bodies`, `frames`, `states`; w-first quaternions; no timestamps). The simulator records it through the `solveit_db` sidecar — an r2d2 connection pool with WAL journal mode and a 5-second busy timeout, which is why the notebook's own plain-Python `sqlite3` cell can read the file while the simulator still holds it open. Build the sidecar once and open the dataset yourself:
```powershell
cd dataset_tools; cargo build --release; cd ..
python -m sqlite3 datasets/nb_dynamic_routh_p2_two_quadrics.db "SELECT name FROM sqlite_master WHERE type='table'"
python -m sqlite3 datasets/nb_dynamic_routh_p2_two_quadrics.db "SELECT COUNT(*) FROM frames"
```
(`python -m sqlite3` is the SQLite shell that ships inside Python 3.12+; the stand-alone `sqlite3` command-line shell, if you have it installed, takes the same statements.)
Inside the notebook the same database answers physics questions through the `sql` command (the COMPLETE SQLite grammar, verbatim), `dataset play` renders its time series to a browser page, and section 10 derives a **movie** from it when there are rigid bodies to animate. Export the transcript (and any paired artifacts) with:
```powershell
python SolveIt_Notebooks_for_rust/03_dynamics_and_routh/dynamic_routh_p2_two_quadrics/player_dynamic_routh_p2_two_quadrics.py data exported_data
```
The paired source scene/script is `dynamic_notebooks/routh_p2_two_quadrics.posim`.

## 5. The full player (companion script, shipped in this folder)

`player_dynamic_routh_p2_two_quadrics.py` sits next to this notebook. It re-executes the notebook (`run`), opens the browser GUI (`gui`), **jumps to the key event** (`jump`), captures a PNG of the GUI headlessly (`capture out.png`, needs Google Chrome or Microsoft Edge), and exports the data (`data outdir`). It is standard-library Python and never computes physics — it drives and displays the recorded, verified artifacts. The complete script, verbatim:

```python
#!/usr/bin/env python3
"""player_dynamic_routh_p2_two_quadrics.py — full player for dynamic_routh_p2_two_quadrics.ipynb (standard library only).

Modes
  python player_dynamic_routh_p2_two_quadrics.py run              re-execute this notebook copy
  python player_dynamic_routh_p2_two_quadrics.py gui              open the browser GUI
  python player_dynamic_routh_p2_two_quadrics.py jump             open the GUI jumped to the key
                                          event (the dynamic scene's final state)
  python player_dynamic_routh_p2_two_quadrics.py capture out.png  save a PNG of the GUI
                                          (headless Chrome or Edge)
  python player_dynamic_routh_p2_two_quadrics.py data outdir      export this notebook's data

Family: dynamic.  Everything runs against recorded, verified artifacts of
the pure-Rust SUNDIALS engine; the player never computes physics itself.
"""

import http.server
import json
import os
import shutil
import socket
import subprocess
import sys
import tempfile
import threading
import time
import webbrowser
from pathlib import Path

HERE = Path(__file__).resolve().parent
WS = HERE.parents[2]                       # workspace root
# Two layouts resolve automatically: a workspace folder holding the engine
# repository beside SolveIt_Notebooks_for_rust/ (either platform edition's
# folder name), or a fresh clone, where the clone root IS the engine.
ENGINE = WS
for _name in ("rustSolveIt_Win11_SUNDIALS_7_8_0",
              "rustSolveIt_macos-silicon_SUNDIALS_7_8_0"):
    if (WS / _name / "posim").is_dir():
        ENGINE = WS / _name
        break
TAG = "dynamic_routh_p2_two_quadrics"
FAMILY = "dynamic"
GUI_PAGE = None                      # main browser-GUI artifact (or None)
MOVIE = None                            # recorded movie page (or None)
GUI_SERVER = None                  # live-GUI server.py (or None)
POSIM_SCRIPT = ENGINE / "dynamic_notebooks/routh_p2_two_quadrics.posim"              # paired .posim scene (or None)
DBS = []                                # SQLite database files
CAPTURE_BUTTON = None          # id of the page's jump button


def chrome():
    """A headless-capable Chromium browser: Google Chrome first (its
    Windows, macOS and Linux install locations), then Microsoft Edge."""
    pf = os.environ.get("ProgramFiles", r"C:\Program Files")
    pf86 = os.environ.get("ProgramFiles(x86)", r"C:\Program Files (x86)")
    local = os.environ.get("LOCALAPPDATA", "")
    for c in (os.path.join(pf, "Google", "Chrome", "Application", "chrome.exe"),
              os.path.join(pf86, "Google", "Chrome", "Application", "chrome.exe"),
              os.path.join(local, "Google", "Chrome", "Application", "chrome.exe")
              if local else "",
              "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome",
              shutil.which("chrome") or "",
              shutil.which("google-chrome") or "",
              shutil.which("chromium") or "",
              os.path.join(pf86, "Microsoft", "Edge", "Application", "msedge.exe"),
              os.path.join(pf, "Microsoft", "Edge", "Application", "msedge.exe")):
        if c and Path(c).exists():
            return c
    raise SystemExit("headless capture needs Google Chrome (or Microsoft Edge) installed")


def free_port():
    s = socket.socket()
    s.bind(("127.0.0.1", 0))
    p = s.getsockname()[1]
    s.close()
    return p


def serve(directory, page_name, jump=False):
    """Serve `directory` on an ephemeral port; return the page URL.
    With jump=True, serve an augmented copy that clicks the page's own
    jump-to-capture button after load (the original file is untouched)."""
    import functools
    port = free_port()
    workdir = directory
    name = page_name
    if jump and CAPTURE_BUTTON:
        tmp = Path(tempfile.mkdtemp(prefix="player_" + TAG + "_"))
        body = (Path(directory) / page_name).read_text(encoding="utf-8")
        body += ('<script>window.addEventListener("load",function(){'
                 'var b=document.getElementById("' + CAPTURE_BUTTON + '");'
                 'if(b){b.click();}});</script>')
        (tmp / name).write_text(body, encoding="utf-8")
        workdir = tmp
    handler = functools.partial(http.server.SimpleHTTPRequestHandler,
                                directory=str(workdir))
    httpd = http.server.ThreadingHTTPServer(("127.0.0.1", port), handler)
    threading.Thread(target=httpd.serve_forever, daemon=True).start()
    return f"http://127.0.0.1:{port}/{name}", httpd


def scene_replay_url():
    """Replay the paired .posim scene into a fresh posim child and open its
    live scene window server; returns (url, process)."""
    binary = os.environ.get("POSIM_BIN", "")
    if not binary:
        # Windows builds posim.exe; probe it before the extensionless name.
        for name in (("posim.exe", "posim") if os.name == "nt" else ("posim",)):
            cand = ENGINE / "target" / "release" / name
            if cand.is_file():
                binary = str(cand)
                break
    if not binary or not Path(binary).is_file():
        raise SystemExit("build the engine first, from the repository root: "
                         "cargo build --release -p posim")
    port = free_port()
    proc = subprocess.Popen([binary, "--machine"],
                            stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                            text=True, bufsize=1,
                            env=dict(os.environ, POSIM_NO_BROWSER="1"))
    def rpc(cmd):
        proc.stdin.write(json.dumps({"op": "exec", "code": cmd}) + "\n")
        proc.stdin.flush()
        while True:
            line = proc.stdout.readline()
            if not line:
                raise RuntimeError("posim closed the connection")
            r = json.loads(line)
            if "event" in r:          # asynchronous scene notice, not a reply
                continue
            if not r.get("ok"):
                raise RuntimeError("posim refused %r: %s" % (cmd, r.get("error")))
            return r
    # Commands are line-based, but DEF blocks span lines: accumulate until
    # braces balance and send each complete statement as one code string.
    pending = []
    depth = 0
    for raw in Path(POSIM_SCRIPT).read_text(encoding="utf-8").splitlines():
        stripped = raw.strip()
        if not pending and (not stripped or stripped.startswith("#")):
            continue
        pending.append(raw)
        depth += raw.count("{") - raw.count("}")
        if depth <= 0:
            rpc("\n".join(pending))
            pending = []
            depth = 0
    if pending:
        rpc("\n".join(pending))
    rpc(f"scene create {port}")
    time.sleep(0.5)
    return f"http://127.0.0.1:{port}/", proc


def open_gui(jump=False):
    if FAMILY in ("solveit", "dynamic", "collision") or (
            FAMILY == "video" and GUI_PAGE is None and MOVIE is None):
        if not POSIM_SCRIPT:
            print("this notebook has no paired .posim scene; its verified")
            print("artifact is the executed notebook itself — open it with:")
            print("  jupyter lab", TAG + ".ipynb")
            return
        url, proc = scene_replay_url()
        print("live posim scene window:", url)
        print("(the paired scene", Path(POSIM_SCRIPT).name,
              "was replayed; Ctrl-C to quit)")
        webbrowser.open(url)
        try:
            proc.wait()
        except KeyboardInterrupt:
            proc.terminate()
        return
    if GUI_SERVER and not jump:
        print("starting the live GUI server (Ctrl-C to quit):", GUI_SERVER)
        subprocess.run([sys.executable, str(GUI_SERVER)], cwd=str(ENGINE))
        return
    page = GUI_PAGE or MOVIE
    if page is None:
        print("this notebook family has no browser GUI; its verified artifact")
        print("is the executed notebook itself — open it with: jupyter lab",
              TAG + ".ipynb")
        return
    url, httpd = serve(str(Path(page).parent), Path(page).name, jump=jump)
    print(("jumped to " + "the dynamic scene's final state" + ": ") if jump else "GUI: ", url)
    webbrowser.open(url)
    print("serving; Ctrl-C to quit")
    try:
        while True:
            time.sleep(3600)
    except KeyboardInterrupt:
        httpd.shutdown()


def capture(out_png, jump=True):
    c = chrome()
    if FAMILY in ("solveit", "dynamic", "collision") or (
            FAMILY == "video" and GUI_PAGE is None and MOVIE is None):
        if not POSIM_SCRIPT:
            raise SystemExit("no paired .posim scene recorded for this notebook")
        url, proc = scene_replay_url()
        try:
            subprocess.run([c, "--headless=new", "--disable-gpu",
                            "--window-size=1500,950",
                            "--screenshot=" + str(out_png), url],
                           check=True, capture_output=True, timeout=120)
        finally:
            proc.terminate()
    else:
        page = GUI_PAGE or MOVIE
        if page is None:
            raise SystemExit("no browser GUI to capture for this notebook")
        url, httpd = serve(str(Path(page).parent), Path(page).name, jump=jump)
        time.sleep(0.3)
        subprocess.run([c, "--headless=new", "--disable-gpu",
                        "--window-size=1500,950", "--virtual-time-budget=8000",
                        "--screenshot=" + str(out_png), url],
                       check=True, capture_output=True, timeout=120)
        httpd.shutdown()
    print("captured:", out_png)


def export_data(outdir):
    out = Path(outdir)
    out.mkdir(parents=True, exist_ok=True)
    for db in DBS:
        import csv
        import sqlite3
        con = sqlite3.connect(db)
        cur = con.cursor()
        tables = [r[0] for r in cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
        for t in tables:
            cur.execute(f"SELECT * FROM {t}")
            cols = [d[0] for d in cur.description]
            rows = cur.fetchall()
            with open(out / f"{Path(db).stem}_{t}.csv", "w", newline="",
                      encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow(cols)
                w.writerows(rows)
        con.close()
        print("exported", len(tables), "tables from", Path(db).name)
    nb = json.loads((HERE / (TAG + ".ipynb")).read_text(encoding="utf-8"))
    with open(out / "transcript.txt", "w", encoding="utf-8") as f:
        for cell in nb["cells"]:
            if cell["cell_type"] != "code":
                continue
            f.write("### In:\n" + "".join(cell["source"]) + "\n### Out:\n")
            for o in cell.get("outputs", []):
                f.write("".join(o.get("text", [])))
            f.write("\n" + "=" * 70 + "\n")
    print("exported executed transcript ->", out / "transcript.txt")
    if MOVIE:
        shutil.copyfile(MOVIE, out / Path(MOVIE).name)
        print("copied movie page ->", out / Path(MOVIE).name)


def main():
    mode = sys.argv[1] if len(sys.argv) > 1 else "gui"
    if mode == "run":
        sys.exit(subprocess.run([sys.executable,
                                 str(WS / "SolveIt_Notebooks_for_rust" / "_tools" / "run_copy.py"),
                                 str(HERE / (TAG + ".ipynb"))]).returncode)
    elif mode == "gui":
        open_gui(jump=False)
    elif mode == "jump":
        open_gui(jump=True)
    elif mode == "capture":
        capture(sys.argv[2] if len(sys.argv) > 2 else TAG + ".png")
    elif mode == "data":
        export_data(sys.argv[2] if len(sys.argv) > 2 else "exported_data")
    else:
        print(__doc__)
        sys.exit(2)


if __name__ == "__main__":
    main()
```

# Routh II.Arts.140-142 — the two quadrics whose intersection IS the polhode

**This notebook is one half of a pair.** The other half is the example
file `dynamic_notebooks/routh_p2_two_quadrics.posim` in this repository, which contains the same simulation
written directly in the simulator's own command language. This notebook
runs that same simulation from Python, and explains every line of it.

**This notebook is complete in itself.** Everything needed to understand
and run it is written below. You will never be asked to open another
file to find an explanation.

Routh II.Arts.140-142 — the two quadrics whose intersection IS the polhode — dynamic notebook

To run the paired example directly, without Python and without Jupyter:

```bash
cargo run -p posim --release -- --notebook dynamic_notebooks/routh_p2_two_quadrics.posim
```

---


## 1. How to open a notebook like this one, from a terminal

This section is repeated in full in every notebook in this project, so
that you never have to go looking in another file for it.

### 1.1 What you need

| you need | why | check it with |
|---|---|---|
| **Rust** (1.75 or newer, MSVC toolchain) | the simulator is pure Rust and is built from source | `rustc --version` |
| **Python 3.8+** | Jupyter runs on Python, and this notebook is a Python notebook | `python --version` |
| **JupyterLab** or **Jupyter Notebook** | the program that opens `.ipynb` files | `jupyter --version` |
| the **posim** binary | the simulator itself; you build it once, below | `dir target\release\posim.exe` |

You do **not** need SUNDIALS installed. You do **not** need a C or Fortran
compiler. You do **not** need any package from crates.io. The whole solver
suite — CVODE, CVODES, IDA, IDAS, KINSOL, ARKODE — is a pure-Rust
translation vendored inside this repository, and `cargo` builds it from
the source that is already on your disk. Nothing is downloaded at build
time.

### 1.2 Build the simulator, once

Open a terminal (PowerShell on Windows) and run these four commands.
`>` is the shell prompt — do not type it.

```powershell
> git clone https://github.com/once-ere/rustSolveIt_Win11_SUNDIALS_7_8_0.git
> cd rustSolveIt_Win11_SUNDIALS_7_8_0
> cargo build --release -p posim
> dir target\release\posim.exe
```

The third command is the long one: it compiles the simulator and the whole
vendored SUNDIALS translation, and takes a few minutes the first time.
When it finishes, the fourth command must print a line describing an
executable file. If it says the file cannot be found, the build did
not succeed — scroll up in the terminal and read the first error, because
later errors are usually consequences of it.

(On Linux or macOS the same commands work with `ls -l
target/release/posim` as the final check; the binary there has no
`.exe` suffix.)

### 1.3 Install Jupyter, if you do not have it

```powershell
> python -m pip install --user jupyterlab
> jupyter --version
```

If `jupyter` is still "not recognized" after this, your `pip --user`
scripts directory is not on your `PATH`. Print it and add it (for the
current PowerShell session):

```powershell
> python -m site --user-site
> $env:PATH = "$(python -m site --user-base)\Scripts;$env:PATH"
```

To make that permanent, add the `...\Scripts` directory to your user
`PATH` in Settings → System → About → Advanced system settings →
Environment Variables. (On Linux/macOS the equivalent is `export
PATH="$(python3 -m site --user-base)/bin:$PATH"` in `~/.bashrc`.)

### 1.4 Start Jupyter and open a NEW notebook

From the repository root (the `rustSolveIt_Win11_SUNDIALS_7_8_0`
directory you cloned above):

```powershell
> jupyter lab
```

That starts a small web server and prints a URL that contains a one-time
token, looking like:

```
    http://localhost:8888/lab?token=8f4c1e...
```

It normally opens your browser by itself. If it does not, copy that whole
URL — token included — into a browser. The token is the password; a URL
without it will be refused.

Then, inside JupyterLab:

1. **File → New → Notebook**.
2. When it asks you to *Select Kernel*, choose **Python 3 (ipykernel)**.
3. The empty notebook appears. Anything you type into a cell is Python.

Choose **Python 3**, and not the "posim" kernel that this repository also
ships, because this notebook is written in Python: it starts the simulator
itself, sends it commands, reads the replies back, and (at the end) opens
a graphical save dialog. Those are Python actions. The posim kernel speaks
only the simulator's own command language and cannot do them.

### 1.5 Running cells

- **Shift+Enter** runs the selected cell and moves to the next one.
- **Ctrl+Enter** runs the selected cell and stays put.
- Run the cells of this notebook **in order, from the top**. Each one
  builds on the state left by the ones above it, so skipping one will
  usually produce an error in a later cell.
- If things get into a confusing state, use **Kernel → Restart Kernel and
  Clear All Outputs**, then start again from the first cell.

### 1.6 If you would rather not use Jupyter at all

Everything this notebook does can also be typed straight into the
simulator's own prompt:

```powershell
> cargo run --release -p posim
```

That gives you an `In[1]:=` prompt where the simulator's commands are
typed directly. Type `HELP` for the command reference and `QUIT` to
leave. The Python in this notebook is a wrapper around exactly that
program.

---


## 2. The words used in this notebook

Repeated in full in every notebook of this project, so that this notebook
stands alone.

**Body.** One rigid object. It has a *shape* (which fixes its moments of
inertia), a mass, a position, a velocity, an orientation and an angular
velocity. Bodies are created with the `NEW` command.

**Boundary / shape.** One of `point`, `sphere`, `cuboid`, `cylinder`,
`disk`, `torus`, `dumbbell`. The shape is not decoration: the simulator
computes the body's inertia tensor from it analytically.

**Static body / anchor.** A body with `inverse_mass = 0`. No force can
move it. It is how a mechanism is bolted to the world. A `point` anchor
additionally has a zero inertia tensor, so no torque can turn it either.

**Joint / constraint.** An exact geometric or kinematic relation that the
solver holds true for all time — not a stiff spring. Each joint
contributes a fixed number of scalar equations, called *rows*:

| command | rows | what it holds | freedoms it leaves |
|---|---|---|---|
| `CONSTRAIN a b [len]` | 1 | a fixed distance between two centres | 5 |
| `GEAR a b <axis> <ratio>` | 1 | a fixed proportion between two turns | 5 |
| `RACK p b <axis> <dir> <r>` | 1 | a turn tied to a slide, `ds = r dtheta` | 5 |
| `BALL a b` | 3 | one shared point | 3 (any rotation about it) |
| `UNIVERSAL a b <u> <w>` | 4 | a shared point and a right angle | 2 |
| `HINGE a b <axis>` | 5 | a shared point and a shared axis | 1 (the swing) |
| `PRISMATIC a b <axis>` | 5 | a line to slide along, and no turning | 1 (the slide) |

**Degrees of freedom.** Each free body has 6 (three of position, three of
orientation). A mechanism of `N` free bodies has `6N`. Subtract the total
rows of all its joints and what remains is how many independent ways the
mechanism can still move. If the subtraction leaves zero the mechanism is
locked; if the rows are not independent of one another the system is
*redundant* and the solver will refuse it rather than guess.

**The pivot rule.** Every joint that holds a point places that point at
the **midpoint of the two bodies' centres, as they stand at the moment the
joint is created**, and then remembers it in each body's own frame. So you
position the parts first and join them second, and the geometry you
assemble is the geometry you get.

**`g` and `g_dot`.** `g` is the vector of constraint equations; the solver
holds `g = 0`. `g_dot` is its time derivative, which the solver also holds
at zero. Their sizes are the total row count. The `CONSTRAINTS` command
prints the worst absolute value of each, and those two numbers are how you
tell whether a mechanism is really being held together: they should stay
near the solver's tolerance and never grow steadily.

**`METHOD IDA`.** Any mechanism with a joint must be integrated by IDA,
the differential-algebraic solver, and the simulator refuses any other
method rather than silently integrating an unconstrained problem.

---


## 3. How a second-order mechanics problem becomes a first-order system

Repeated in full in every notebook of this project.

SUNDIALS — like every general ODE/DAE library — integrates **first-order**
systems. Newton's and Euler's equations are **second order**. This section
is the bridge, and it is the same bridge in every example.

### 3.1 The physics, as written by hand

For each body `i`, with mass `m_i`, inertia tensor `I_i` (in the body's own
frame), position `x_i`, orientation `R_i`, and angular velocity `omega_i`:

```
m_i * d2(x_i)/dt2 = F_i                            (Newton)
I_i * d(omega_i)/dt + omega_i x (I_i omega_i) = T_i   (Euler)
```

Both are second order in the configuration variables: the first is second
order in `x_i`, and the second is second order in orientation, because
`omega_i` is itself the rate of change of `R_i`.

### 3.2 The standard trick: name the velocities

A second-order equation becomes two first-order equations by giving the
first derivative its own name and letting the solver carry it as an
unknown alongside the position. This simulator carries **momentum** rather
than velocity, which is the same trick with a better-conditioned variable:

```
d(x_i)/dt = p_i / m_i                    <- was the definition of velocity
d(p_i)/dt = F_i                          <- was Newton's second law
d(q_i)/dt = 0.5 * quat(omega_i) * q_i    <- was the definition of omega
d(L_i)/dt = T_i                          <- was Euler's equation
```

with `omega_i = R_i I_i^-1 R_i^T L_i`, and `q_i` a unit quaternion holding
the orientation. Orientation is stored as a quaternion rather than as
Euler angles because quaternions have no gimbal lock and renormalise
cheaply.

### 3.3 The packed state vector

Each body contributes exactly **13 numbers**, in this order:

```
index 0..2    x       position
index 3..5    p       linear momentum
index 6..9    q       orientation quaternion, w first
index 10..12  L       angular momentum
```

so a model of `N` bodies has a state vector `y` of length `13N`, and the
whole of mechanics above is one first-order system `dy/dt = f(t, y)`.
That is what is handed to the Rust translation of CVODE or ARKODE.

### 3.4 When there are joints: a DAE, not an ODE

A joint is an algebraic relation between coordinates, not a rate, so it
cannot be written as `dy/dt = ...`. The system becomes *differential-
algebraic*, and this simulator uses the **GGL (Gear-Gupta-Leimkuhler)**
index-2 formulation, which carries **both** the position-level constraint
and its derivative, with a Lagrange multiplier for each:

```
d(x)/dt = v - M^-1 J^T mu        position, corrected by the multiplier mu
d(p)/dt = F + J^T lambda         momentum, plus the constraint force
0       = g(q)                   the joints hold, at the position level
0       = J u                    the joints hold, at the velocity level
```

`J` is the constraint Jacobian: row `k` of `J` is the gradient of the
`k`-th constraint equation with respect to all the coordinates, so `J u`
is the rate of change of the constraints under the current velocities.
`lambda` are the constraint forces, `mu` the position-level correction,
and `M^-1` the inverse mass/inertia metric — which is **not** optional,
because `J` has rows in different units and the correction has to be
mass-weighted to be dimensionally consistent.

Carrying both `g` and `J u` is what keeps the joints exact. A formulation
that enforced only the acceleration level would let `g` drift away
quadratically in time, and a mechanism would slowly come apart.

The unknowns are therefore `y` (the `13N` numbers above) together with
`lambda` and `mu` (one of each per constraint row), and the whole thing is
handed to the Rust translation of **IDA**, which solves systems of the
implicit form `F(t, y, y') = 0`.

### 3.5 Consistent initial conditions

A DAE cannot be started from just any state. The starting state must
already satisfy `g = 0` **and** `J u = 0` — the parts must be assembled,
and their initial velocities must be compatible with the joints. If the
velocities are not compatible, this simulator projects them onto the
nearest compatible set before starting, and reports how much it had to
change them. A start that needs no projection is one where you specified
velocities the mechanism can actually have.

---


## 4. The physical situation

### The objects

| name | shape | given properties |
|---|---|---|
| `obj0` | `cuboid` | `mass = 3`, `half_extents = [0.5, 1, 2]`, `angular_velocity = [0.3, 3, 0.1]` |
| `obj1` | `cuboid` | `mass = 3`, `half_extents = [0.5, 1, 2]`, `angular_velocity = [0.3, 3, 0.1]` |

Each row is one rigid body. The simulator computes each body's inertia tensor from its shape; you never type an inertia tensor in.

### The shapes, and the inertia each implies

**`cuboid`** — a uniform rectangular block. `I = (m/3) * diag(b^2+c^2, a^2+c^2, a^2+b^2)` for half-extents `(a, b, c)`. Three distinct moments in general, which is what makes a cuboid able to tumble. Half-extents are measured from the centre, so the full side lengths are twice these numbers.

### What each property means

- **`mass`** — the body's mass, in kilograms. Setting it also sets `inverse_mass` to its reciprocal — the two are kept consistent, and the simulator carries the inverse because that is what the equations of motion actually use.
- **`half_extents`** — half the side lengths, as `[a, b, c]`, measured from the centre. The full block is `2a` by `2b` by `2c`.
- **`angular_velocity`** — the spin rate `omega`, as `[wx, wy, wz]` in radians per second, in WORLD axes. Setting it sets the angular momentum to `L = I omega`, which is what is carried.

### The interactions

- **Mutual gravity is OFF** (`g_constant = 0`). The simulator's default is 1, so this had to be said explicitly; without it, bodies that are supposed only to bounce or to be held by joints would also attract each other.
- **Contact is OFF**. Nothing here is meant to touch anything else, so the rootfinder is not armed.

### 4.1 Equations of motion

There are 2 bodies. Each obeys Newton's and Euler's equations, both of which are **second order**:

```
m_i * d2(x_i)/dt2 = 0

I_i * d(omega_i)/dt + omega_i x (I_i omega_i) = T_i
```

The `omega x (I omega)` term in Euler's equation is the gyroscopic term. It is what makes a tumbling body's spin axis wander even with no torque at all, and it is why rigid-body motion cannot be reduced to three independent rotations.

The forces on the right are:

- nothing: every body coasts freely.

### 4.2 Constraint equations

**This model has no joints.** Nothing algebraic relates the coordinates, so the system is an ordinary differential equation rather than a differential-algebraic one, and it needs no Lagrange multipliers and no constraint Jacobian.

That does not mean nothing is conserved. Energy, linear momentum and angular momentum are still exact consequences of the equations of motion, and comparing them before and after a run is the honest way to judge the integration.

### 4.3 The first-order system actually handed to SUNDIALS

**Sizing this particular problem.** There are 2 bodies, so the state vector `y` has

```
2 bodies x 13 numbers = 26 components
```

laid out as consecutive 13-number blocks in creation order, each holding position (3), linear momentum (3), orientation quaternion w-first (4), and angular momentum (3).

**What is handed to the Rust SUNDIALS translation.**

There are no algebraic constraints, so the whole of the mechanics above is one first-order system

```
dy/dt = f(t, y),   y in R^26
```

handed to the pure-Rust translation of **CVODE** in Adams-Moulton mode: a variable-order, variable-step explicit-family method for non-stiff problems. This is the default, and it is the right choice for orbits and for anything smooth.

The right-hand side `f` unpacks `y` into bodies, computes every force and torque, and packs the derivatives back in the same 13-per-body order.

---


## 5. How this notebook talks to the simulator

Repeated in full in every notebook of this project.

The next cell defines a small helper class. It launches the simulator as a
child process in **machine mode** (`posim --machine`), which makes it speak
JSON Lines: you send it one JSON object per line, and it answers with one
JSON object per line. This is the same interface the project's Jupyter
kernel uses internally.

The three calls used below are:

- `sim.do("...")` — run one simulator command and show what it printed.
  Adding `quiet=True` runs it without printing, which is what you want
  inside a loop that takes thousands of steps.
- `sim.get("obj.field")` — read one value back as a Python object.
- `sim.state()` — fetch the whole simulation state as a Python dictionary.

Nothing here integrates anything: every advance in time is performed by
the Rust SUNDIALS translation inside the child process.


In [1]:
import json, os, shutil, subprocess, sys
from pathlib import Path

def _find_posim():
    """Locate the posim binary, explaining clearly if it is missing."""
    env = os.environ.get("POSIM_BIN")
    if env and Path(env).is_file():
        return env
    onpath = shutil.which("posim")
    if onpath:
        return onpath
    here = Path.cwd()
    names = ("posim.exe", "posim") if os.name == "nt" else ("posim",)
    for base in [here, *here.parents]:
        for profile in ("release", "debug"):
            for name in names:
                cand = base / "target" / profile / name
                if cand.is_file():
                    return str(cand)
    raise SystemExit(
        "Could not find the posim binary.\n"
        "Build it first, from the repository root:\n"
        "    cargo build --release -p posim\n"
        "or set the POSIM_BIN environment variable to its full path."
    )

class Sim:
    """One posim child process, spoken to in JSON Lines."""

    def __init__(self):
        self.binary = _find_posim()
        self.proc = subprocess.Popen(
            [self.binary, "--machine"],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE,
            text=True, encoding="utf-8", bufsize=1,
            env=dict(os.environ, POSIM_NO_BROWSER="1"),
        )
        # Show where the binary is RELATIVE to the notebook, not its absolute
        # location. The absolute form names the machine that ran the notebook
        # (username and full directory tree) and gets baked into every
        # committed output cell; the relative form is both portable and the
        # more useful thing to read.
        try:
            shown = os.path.relpath(self.binary)
        except ValueError:          # different drive: no relative path exists
            shown = os.path.basename(self.binary)
        print(f"simulator started: {shown}")

    def _rpc(self, obj):
        self.proc.stdin.write(json.dumps(obj) + "\n")
        self.proc.stdin.flush()
        while True:
            line = self.proc.stdout.readline()
            if not line:
                raise RuntimeError("the simulator closed the connection")
            reply = json.loads(line)
            if "event" in reply:      # asynchronous scene notice, not a reply
                continue
            return reply

    def do(self, code, quiet=False):
        """Run one simulator command; print and return what it printed.

        Pass quiet=True inside a loop, where printing a line per step
        would bury the result under thousands of lines of progress.
        """
        r = self._rpc({"op": "exec", "code": code})
        if not r.get("ok"):
            raise RuntimeError(f"the simulator refused {code!r}:\n  {r.get('error')}")
        out = r.get("display") or r.get("result")
        if out not in (None, "") and not quiet:
            print(out)
        return out

    def get(self, path):
        """Read one value, e.g. sim.get('piston.position')."""
        r = self._rpc({"op": "get", "path": path})
        if not r.get("ok"):
            raise RuntimeError(f"could not read {path!r}: {r.get('error')}")
        return r.get("result")

    def state(self):
        """The whole simulation state, as a Python dictionary."""
        r = self._rpc({"op": "state"})
        if not r.get("ok"):
            raise RuntimeError(r.get("error"))
        return r["result"]

    def close(self):
        try:
            self.proc.stdin.write('{"op":"quit"}\n'); self.proc.stdin.flush()
        except (BrokenPipeError, ValueError):
            pass
        self.proc.wait(timeout=30)
        for pipe in (self.proc.stdin, self.proc.stdout):
            try: pipe.close()
            except Exception: pass
        print("simulator stopped")

sim = Sim()

simulator started: ..\..\..\target\release\posim.exe


## 6. This notebook's dataset

Every notebook in this repository owns a **dataset**: one SQLite
database file, created fresh each time the notebook runs, that records
the simulation as it happens. This notebook's dataset is
`datasets/nb_dynamic_routh_p2_two_quadrics.db`, and the cell below creates it.

What "recording" means here: from the moment the dataset is created,
every output row of a `run` and the end of every `step` is appended as
one **frame** — the time, the total energy, the momentum and
angular-momentum components, the centre of mass, the cumulative
collision count, and the full position / orientation (w-first
quaternion) / velocity / angular-velocity state of every body. The
schema has four tables: `meta` (key/value pairs), `bodies` (one row per
body: name, shape, mass, geometry), `frames` (one row per recorded
instant) and `states` (one row per body per frame).

The simulator itself contains no SQLite code at all. It drives a small
sidecar program, `solveit_db` (built once with
`cd dataset_tools; cargo build --release; cd ..` from the repository root),
which owns a thread-safe connection pool (`r2d2`), switches the
database into **WAL journal mode** at startup, and sets a five-second
**busy timeout** — the standard recipe that lets several readers and a
writer share one SQLite file without ever seeing `database is locked`.
WAL matters to *this notebook* concretely: a later cell reads the
database with Python's own `sqlite3` module **while the simulator still
holds it open**, which is exactly the cross-process concurrency WAL
provides.

Creating a dataset arms recording automatically (`dataset record off`
would pause it). If a dataset of this name already exists from an
earlier run of this notebook, it is replaced, so a re-run always
reproduces the same database.


In [2]:
sim.do("dataset create nb_dynamic_routh_p2_two_quadrics")

dataset nb_dynamic_routh_p2_two_quadrics created (datasets/nb_dynamic_routh_p2_two_quadrics.db, journal_mode=wal, schema v1) — recording armed


## 7. Building and running the simulation

From here on, the notebook alternates: a section of text explaining
exactly what the next cell asks the simulator to do and what each value
in it means, then the cell itself. Run them in order from here down.


### 7.1 Set up the world

The next cell sends these 3 commands to the simulator. Here is exactly what each one expects:

---

`set system.g_constant = 0` sets a property of the whole simulation rather than of one body. `g_constant` is the gravitational constant `G` used for the mutual attraction between every pair of bodies, `F = G m1 m2 / r^2`. **Its default is 1, not 0**, which catches people out: bodies rattling in a box also attract each other unless you say otherwise. Set it to 0 for a mechanism or a collision experiment; set it to a real value for an orbit.

The value expected here is `0`. A successful `set` prints nothing.

---

`set system.rtol = 1e-13` sets a property of the whole simulation rather than of one body. `rtol` is the integrator's relative error tolerance. Smaller is more accurate and slower. Joints that grip orientation impose a floor of `1e-6`, because the index-2 accuracy ceiling is real and asking for more produces convergence failures rather than better answers.

The value expected here is `1e-13`. A successful `set` prints nothing.

---

`set system.atol = 1e-15` sets a property of the whole simulation rather than of one body. `atol` is the integrator's absolute error tolerance, which is what governs components passing near zero, where a relative tolerance means nothing. Orientation joints impose a floor of `1e-8`.

The value expected here is `1e-15`. A successful `set` prints nothing.

In [3]:
sim.do("set system.g_constant = 0")
sim.do("set system.rtol = 1e-13")
sim.do("set system.atol = 1e-15")

### 7.2 Create the bodies

The next cell sends this command to the simulator. Here is exactly what it expects:

---

`new cuboid` creates one rigid body whose shape is **a uniform rectangular block**, left unnamed, so it takes the next automatic name `objN`. The general form is

```
new <shape> [as <name>] { <field> = <value>, ... }
```

Its inertia tensor is not typed in — the simulator computes it from the shape: `I = (m/3) * diag(b^2+c^2, a^2+c^2, a^2+b^2)` for half-extents `(a, b, c)`. Three distinct moments in general, which is what makes a cuboid able to tumble. Half-extents are measured from the centre, so the full side lengths are twice these numbers.

The fields given here are:

- `mass = 3` — the body's mass, in kilograms. Setting it also sets `inverse_mass` to its reciprocal — the two are kept consistent, and the simulator carries the inverse because that is what the equations of motion actually use.
- `half_extents = [0.5, 1, 2]` — half the side lengths, as `[a, b, c]`, measured from the centre. The full block is `2a` by `2b` by `2c`.
- `angular_velocity = [0.3, 3, 0.1]` — the spin rate `omega`, as `[wx, wy, wz]` in radians per second, in WORLD axes. Setting it sets the angular momentum to `L = I omega`, which is what is carried.

The command prints the name the simulator assigned, which is how you confirm the body exists.

In [4]:
sim.do("new cuboid { mass = 3, half_extents = [0.5, 1, 2], angular_velocity = [0.3, 3, 0.1] }")

obj0


### 7.3 Measure

The next cell sends these 4 commands to the simulator. Here is exactly what each one expects:

---

`get` evaluates an expression and prints it. The form is `GET <expression>`, where an expression may name a body's field (`obj0.position`, `bob.velocity.x`), index into it, or combine several with arithmetic and the built-in functions `norm`, `normalize`, `dot`, `cross`, `sqrt` and the rest.

Here it evaluates:

```
obj0.inertia_tensor
```

The corresponding Python call `sim.get(...)` returns the value as a Python object instead of printing it, which is what you want when the number is about to be used in a calculation.

---

`get` evaluates an expression and prints it. The form is `GET <expression>`, where an expression may name a body's field (`obj0.position`, `bob.velocity.x`), index into it, or combine several with arithmetic and the built-in functions `norm`, `normalize`, `dot`, `cross`, `sqrt` and the rest.

Here it evaluates:

```
obj0.angular_momentum
```

The corresponding Python call `sim.get(...)` returns the value as a Python object instead of printing it, which is what you want when the number is about to be used in a calculation.

---

`norm(obj0.angular_momentum)` calls a previously defined function with the arguments given. A call that fails part way through leaves no half-built body behind.

---

`energy` prints the total kinetic plus potential energy of the whole system. Take it before and after a run: for a closed system with no dissipation the two should agree to near machine precision, and how well they agree is the honest measure of the integration. It takes no arguments.

In [5]:
sim.do("get obj0.inertia_tensor")
sim.do("get obj0.angular_momentum")
sim.do("norm(obj0.angular_momentum)")
sim.do("energy")

[[5, 0, 0], [0, 4.25, 0], [0, 0, 1.25]]
[1.5, 12.75, 0.125]
12.838540610209558
19.356250000000003


### 7.4 Measure (2)

The next cell sends this command to the simulator. Here is exactly what it expects:

---

`norm(obj0.angular_velocity)` calls a previously defined function with the arguments given. A call that fails part way through leaves no half-built body behind.

In [6]:
sim.do("norm(obj0.angular_velocity)")

3.0166206257996713


### 7.5 Integrate

The next cell sends this command to the simulator. Here is exactly what it expects:

---

`run 3` advances the simulation by **3 seconds of duration** — not to absolute time 3. This trips people up: `run 1.7` followed by `run 2.89` leaves you at t = 4.59, not at 2.89. The form is `RUN <duration> [STEPS <n>]`.

`steps 3` asks for the result to be reported at 3 equally spaced points across that interval. It does not set the integrator's internal step size — SUNDIALS chooses that itself from the error tolerances, and takes as many internal steps between reports as accuracy demands.

This is the cell that actually integrates, so it is the slow one. It prints the time reached and how many internal solver steps were needed.

In [7]:
sim.do("run 3 steps 3")

t = 3 (321 solver steps, 3 snapshots, |dE/E| = 7.782e-14)


### 7.6 Measure (3)

The next cell sends these 3 commands to the simulator. Here is exactly what each one expects:

---

`norm(obj0.angular_momentum)` calls a previously defined function with the arguments given. A call that fails part way through leaves no half-built body behind.

---

`energy` prints the total kinetic plus potential energy of the whole system. Take it before and after a run: for a closed system with no dissipation the two should agree to near machine precision, and how well they agree is the honest measure of the integration. It takes no arguments.

---

`norm(obj0.angular_velocity)` calls a previously defined function with the arguments given. A call that fails part way through leaves no half-built body behind.

In [8]:
sim.do("norm(obj0.angular_momentum)")
sim.do("energy")
sim.do("norm(obj0.angular_velocity)")

12.838540610209558
19.356249999998496
3.0376911727373113


### 7.7 Integrate (2)

The next cell sends this command to the simulator. Here is exactly what it expects:

---

`run 3` advances the simulation by **3 seconds of duration** — not to absolute time 3. This trips people up: `run 1.7` followed by `run 2.89` leaves you at t = 4.59, not at 2.89. The form is `RUN <duration> [STEPS <n>]`.

`steps 3` asks for the result to be reported at 3 equally spaced points across that interval. It does not set the integrator's internal step size — SUNDIALS chooses that itself from the error tolerances, and takes as many internal steps between reports as accuracy demands.

This is the cell that actually integrates, so it is the slow one. It prints the time reached and how many internal solver steps were needed.

In [9]:
sim.do("run 3 steps 3")

t = 6 (298 solver steps, 3 snapshots, |dE/E| = 3.452e-13)


### 7.8 Measure (4)

The next cell sends these 3 commands to the simulator. Here is exactly what each one expects:

---

`norm(obj0.angular_momentum)` calls a previously defined function with the arguments given. A call that fails part way through leaves no half-built body behind.

---

`energy` prints the total kinetic plus potential energy of the whole system. Take it before and after a run: for a closed system with no dissipation the two should agree to near machine precision, and how well they agree is the honest measure of the integration. It takes no arguments.

---

`norm(obj0.angular_velocity)` calls a previously defined function with the arguments given. A call that fails part way through leaves no half-built body behind.

In [10]:
sim.do("norm(obj0.angular_momentum)")
sim.do("energy")
sim.do("norm(obj0.angular_velocity)")

12.838540610209558
19.356249999991814
3.406947472501729


### 7.9 Integrate (3)

The next cell sends this command to the simulator. Here is exactly what it expects:

---

`run 3` advances the simulation by **3 seconds of duration** — not to absolute time 3. This trips people up: `run 1.7` followed by `run 2.89` leaves you at t = 4.59, not at 2.89. The form is `RUN <duration> [STEPS <n>]`.

`steps 3` asks for the result to be reported at 3 equally spaced points across that interval. It does not set the integrator's internal step size — SUNDIALS chooses that itself from the error tolerances, and takes as many internal steps between reports as accuracy demands.

This is the cell that actually integrates, so it is the slow one. It prints the time reached and how many internal solver steps were needed.

In [11]:
sim.do("run 3 steps 3")

t = 9 (326 solver steps, 3 snapshots, |dE/E| = 5.369e-13)


### 7.10 Measure (5)

The next cell sends these 4 commands to the simulator. Here is exactly what each one expects:

---

`norm(obj0.angular_momentum)` calls a previously defined function with the arguments given. A call that fails part way through leaves no half-built body behind.

---

`energy` prints the total kinetic plus potential energy of the whole system. Take it before and after a run: for a closed system with no dissipation the two should agree to near machine precision, and how well they agree is the honest measure of the integration. It takes no arguments.

---

`norm(obj0.angular_velocity)` calls a previously defined function with the arguments given. A call that fails part way through leaves no half-built body behind.

---

`reset` returns the simulation to t = 0 and to the state the bodies were created in. It takes no arguments.

In [12]:
sim.do("norm(obj0.angular_momentum)")
sim.do("energy")
sim.do("norm(obj0.angular_velocity)")
sim.do("reset")

12.838540610209558
19.356249999981422
3.178273033435918
system reset


### 7.11 Set up the world (2)

The next cell sends these 3 commands to the simulator. Here is exactly what each one expects:

---

`set system.g_constant = 0` sets a property of the whole simulation rather than of one body. `g_constant` is the gravitational constant `G` used for the mutual attraction between every pair of bodies, `F = G m1 m2 / r^2`. **Its default is 1, not 0**, which catches people out: bodies rattling in a box also attract each other unless you say otherwise. Set it to 0 for a mechanism or a collision experiment; set it to a real value for an orbit.

The value expected here is `0`. A successful `set` prints nothing.

---

`set system.rtol = 1e-13` sets a property of the whole simulation rather than of one body. `rtol` is the integrator's relative error tolerance. Smaller is more accurate and slower. Joints that grip orientation impose a floor of `1e-6`, because the index-2 accuracy ceiling is real and asking for more produces convergence failures rather than better answers.

The value expected here is `1e-13`. A successful `set` prints nothing.

---

`set system.atol = 1e-15` sets a property of the whole simulation rather than of one body. `atol` is the integrator's absolute error tolerance, which is what governs components passing near zero, where a relative tolerance means nothing. Orientation joints impose a floor of `1e-8`.

The value expected here is `1e-15`. A successful `set` prints nothing.

In [13]:
sim.do("set system.g_constant = 0")
sim.do("set system.rtol = 1e-13")
sim.do("set system.atol = 1e-15")

### 7.12 Create the bodies (2)

The next cell sends this command to the simulator. Here is exactly what it expects:

---

`new cuboid` creates one rigid body whose shape is **a uniform rectangular block**, left unnamed, so it takes the next automatic name `objN`. The general form is

```
new <shape> [as <name>] { <field> = <value>, ... }
```

Its inertia tensor is not typed in — the simulator computes it from the shape: `I = (m/3) * diag(b^2+c^2, a^2+c^2, a^2+b^2)` for half-extents `(a, b, c)`. Three distinct moments in general, which is what makes a cuboid able to tumble. Half-extents are measured from the centre, so the full side lengths are twice these numbers.

The fields given here are:

- `mass = 3` — the body's mass, in kilograms. Setting it also sets `inverse_mass` to its reciprocal — the two are kept consistent, and the simulator carries the inverse because that is what the equations of motion actually use.
- `half_extents = [0.5, 1, 2]` — half the side lengths, as `[a, b, c]`, measured from the centre. The full block is `2a` by `2b` by `2c`.
- `angular_velocity = [0.3, 3, 0.1]` — the spin rate `omega`, as `[wx, wy, wz]` in radians per second, in WORLD axes. Setting it sets the angular momentum to `L = I omega`, which is what is carried.

The command prints the name the simulator assigned, which is how you confirm the body exists.

In [14]:
sim.do("new cuboid { mass = 3, half_extents = [0.5, 1, 2], angular_velocity = [0.3, 3, 0.1] }")

obj0


### 7.13 The graphical scene window

The next cell sends these 2 commands to the simulator. Here is exactly what each one expects:

---

`scene create` opens the graphical scene window. It starts a small HTTP and WebSocket server — hand-rolled on `std::net`, with no web framework — and prints the address to open in a browser. The window shows the bodies as wireframes, with a readout of energy, momentum and angular momentum that updates live.

The window evolves its **own copy** of the system. Stepping in the notebook does not move the window, and playing the window does not move the notebook. That isolation is deliberate.

If the environment variable `POSIM_NO_BROWSER` is set, the address is printed but no browser is launched — which is what happens when this notebook is run non-interactively.

---

`scene set_time_step 0.03` controls the graphical scene window. The scene command family covers `CREATE`, `START`, `PAUSE`, `REVERSE`, `RESET`, `REFRESH`, and the camera controls.

In [15]:
sim.do("scene create")
sim.do("scene set_time_step 0.03")

scene window created: http://127.0.0.1:52574/
(no browser was launched — open that address yourself)
showing 1 entity; SCENE START begins the evolution — HELP lists all scene commands
scene time step dt = 0.03


## 8. What those numbers mean

**What the cells above actually established.**

- `energy` was printed. Compare the value before the run with the value after it. For a closed system with no dissipation they should agree to near machine precision, and the size of the disagreement is the honest measure of the integration — not a number to be talked around.

**How to judge a number here.** A disagreement with theory is only a defect if it fails to shrink when you ask for more accuracy. Tighten `system.rtol` and `system.atol` and re-run: a discretisation error converges, and a bug does not. The one place this does not apply is a model with orientation-gripping joints, where `rtol` is floored at `1e-6` and tightening past it buys nothing.

**The full note from the example this notebook pairs with**, which sets out why it was built this way:

Routh II.Arts.140-142 — the two quadrics whose intersection IS the polhode
— dynamic notebook

E. J. Routh, "The Advanced Part of a Treatise on the Dynamics of a System of
Rigid Bodies", Part II (Macmillan, 6th ed. 1905), Art. 140 gives the two
fundamental equations of a body turning under no forces,

A^2 w1^2 + B^2 w2^2 + C^2 w3^2 = G^2        (angular momentum)
A   w1^2 + B   w2^2 + C   w3^2 = T          (vis viva)

and Art. 142 confirms the second is constant because no forces act. Two
quadric surfaces in the space of angular velocities; the motion is confined
to BOTH, so the instantaneous axis runs along their curve of intersection --
the polhode of Art. 149. Everything Poinsot draws is a picture of these two
equations.

WHAT THIS NOTEBOOK SHOWS. That the two quadrics really are invariant while
the thing they constrain is not. A tumbling body is set going and three
numbers are printed repeatedly:

G = |L|              must not move
E = T/2              must not move   (posim's energy is half Routh's T)
|w|                  moves a great deal

It is easy to assume the angular velocity is "the" conserved thing in a
torque-free motion. It is not; neither its direction nor its magnitude is
conserved. Only the two quadrics are, and the wandering of w inside them is
the entire subject of Chapter IV.

THE NUMBERS. A cuboid of mass 3 with half-extents (0.5, 1, 2) gives
I = diag(5, 4.25, 1.25). Started with w = (0.3, 3, 0.1),

L   = I w = (1.5, 12.75, 0.125)
G   = |L| = 12.838540610209558
T   = A w1^2 + B w2^2 + C w3^2 = 38.7125     (Routh's T)
E   = T/2 = 19.35625                         (what posim's ENERGY prints)
|w| at t = 0 = 3.0166206257996713            (this one will change)

The disturbance is deliberately asymmetric so the motion sits clearly off the
separatrix G^2 = B*T of Art. 184 -- here G^2 - B*T = +0.3 -- and the body
genuinely tumbles rather than creeping along the separatrix.

SETUP NOTE. system.g_constant = 0. Only one body is present so gravity would
do nothing anyway, but leaving G = 1 in a multi-body version of this test is
how a purely rotational result gets quietly ruined by a translational
singularity.

WHAT YOU WATCH: a box tumbling irregularly end over end. The hud's E and L
do not move at all while it does.
---------------------------------------------------------------------------
LAUNCH (run from the repository root)

posim_notebook routh_p2_two_quadrics

or, without the shell alias, the always-available form:

cargo run -p posim --release -- --notebook dynamic_notebooks/routh_p2_two_quadrics.posim

The scene window opens in your browser. To start the simulation, press Start
in that window, or type this at the In[] prompt in the terminal:

scene start

If no browser should be launched (the URL is printed instead), use:

POSIM_NO_BROWSER=1 cargo run -p posim --release -- --notebook dynamic_notebooks/routh_p2_two_quadrics.posim
---------------------------------------------------------------------------


## 9. Reading the dataset back

One of the steps above was a `reset` — the command that clears the
world and starts a fresh scenario. A dataset describes ONE world, so
the `reset` checkpointed and closed it; everything recorded up to that
moment is safely on disk in `datasets/nb_dynamic_routh_p2_two_quadrics.db`, and the frames
that follow a reset belong to a different world and are deliberately
not mixed into the same tables.

The cell below reopens the file and interrogates it: `dataset open`
attaches it again, the `sql` command (which accepts ANY SQLite
statement, verbatim — the whole SQLite grammar) counts what was
recorded, and `dataset play` renders the recorded time series into a
self-contained page `datasets/nb_dynamic_routh_p2_two_quadrics_plots.html`, opened in your
web browser (suppressed in batch runs). If nothing had been recorded
before the reset, a snapshot of the current world is taken first so
the plots have at least one frame to show.


In [16]:
print(sim.do("dataset open nb_dynamic_routh_p2_two_quadrics", quiet=True))
frames = int(sim.do('sql "SELECT COUNT(*) FROM frames"', quiet=True).splitlines()[-1])
if frames == 0:
    sim.do("dataset snapshot")
sim.do('sql "SELECT COUNT(*) AS frames, MIN(t) AS t_first, MAX(t) AS t_last FROM frames"')
sim.do('sql "SELECT COUNT(*) AS body_state_rows FROM states"')
sim.do("dataset play")

dataset nb_dynamic_routh_p2_two_quadrics opened (1 bodies, 9 frames, 9 states) — recording armed
frames | t_first | t_last
9 | 1 | 9
body_state_rows
9
dataset plots written to datasets\nb_dynamic_routh_p2_two_quadrics_plots.html (browser suppressed)


The cell below is the third reader: plain Python, using only the
standard library's `sqlite3` module — no simulator involved. It opens
the very database the simulator is still holding open (WAL mode is what
makes that safe), prints the row counts and the last recorded frame,
and closes. Anything a spreadsheet, a plotting library or another
program can do with a SQLite file, it can do with this dataset.


In [17]:
import sqlite3

con = sqlite3.connect("datasets/nb_dynamic_routh_p2_two_quadrics.db")
try:
    tables = {}
    for table in ("meta", "bodies", "frames", "states"):
        tables[table] = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print("row counts:", tables)
    last = con.execute(
        "SELECT frame, t, energy, ncollisions FROM frames ORDER BY frame DESC LIMIT 1"
    ).fetchone()
    if last is not None:
        print(f"last frame: frame={last[0]} t={last[1]} energy={last[2]} collisions={last[3]}")
    for row in con.execute(
        "SELECT b.name, b.shape, COUNT(s.frame) FROM bodies b "
        "LEFT JOIN states s ON s.body = b.body GROUP BY b.body ORDER BY b.body"
    ):
        print(f"body {row[0]!r} ({row[1]}): {row[2]} recorded states")
finally:
    con.close()

row counts: {'meta': 2, 'bodies': 1, 'frames': 9, 'states': 9}
last frame: frame=8 t=9.0 energy=19.356249999981422 collisions=0
body 'obj0' (cuboid): 9 recorded states


## 10. The movie derived from this dataset

A **movie** in this repository is not a screen recording — it is a
self-contained 3-D player page generated FROM a dataset: `movie create
<dataset>` reads the `bodies`, `frames` and `states` tables and writes
`movies/<name>.html`, which animates the recorded rigid bodies with a
scrub bar, play/pause, speed control, an orbiting camera and a live
readout of t, E, |P| and |L|. Because the dataset is the source of
truth, a movie can be deleted and rebuilt at any time without
re-running any physics, and one dataset can carry several movies.

The cell below asks the database how much it recorded, and builds and
plays the movie only if there is something to animate — a dataset with
no moving bodies (for example a purely quantum-mechanical notebook,
whose wavefunction lives on a grid rather than in rigid-body states)
gets an honest explanation instead of an error.


In [18]:
frames = int(sim.do('sql "SELECT COUNT(*) FROM frames"', quiet=True).splitlines()[-1])
state_rows = int(sim.do('sql "SELECT COUNT(*) FROM states"', quiet=True).splitlines()[-1])
if frames >= 2 and state_rows > 0:
    print(sim.do("movie create nb_dynamic_routh_p2_two_quadrics", quiet=True))
    print(sim.do("movie play nb_dynamic_routh_p2_two_quadrics", quiet=True))
else:
    print(f"this dataset holds {frames} frame(s) and {state_rows} body-state row(s);")
    print("a movie needs at least 2 frames of at least one body, so none is made here.")

movie nb_dynamic_routh_p2_two_quadrics created from dataset nb_dynamic_routh_p2_two_quadrics (movies\nb_dynamic_routh_p2_two_quadrics.html, 9 frames, 1 bodies)
movie nb_dynamic_routh_p2_two_quadrics (movies\nb_dynamic_routh_p2_two_quadrics.html) ready (browser suppressed)


## 11. Close the simulator

The simulator is a separate operating-system process that this notebook
started. Closing it releases that process and its two pipes (an open
dataset is checkpointed and closed first, so the SQLite file on disk is
left as one tidy `.db`). If you skip this cell the process is closed
anyway when the Jupyter kernel shuts down, but doing it explicitly means
you can re-run the notebook from the top without leaving a stray process
behind each time.


In [19]:
if not sim.do("dataset record", quiet=True).startswith("no dataset"):
    sim.do("dataset close")
sim.close()

dataset nb_dynamic_routh_p2_two_quadrics closed
simulator stopped


## 12. Name this notebook, and choose where to save it

Repeated in full in every notebook of this project.

The next cell asks you for two things:

1. **A name for this notebook.** It is typed into a normal input box that
   appears just under the cell. Press Enter to accept the suggestion shown
   in brackets.
2. **A folder to save it in.** A graphical "Save As" window opens on top of
   your other windows. Pick a folder, adjust the file name if you like, and
   press Save.

The graphical window is drawn by `tkinter`, which is part of Python's own
standard library, so nothing needs installing. If you are running this
notebook on a machine with no graphical display — over a plain SSH
connection, for instance — that window cannot open. The cell detects this,
says so, and falls back to asking you to type the folder path instead. It
will not fail or hang.

Saving writes a **copy** of this notebook, exactly as it stands now,
including every output you have produced. The original file is left
untouched.


In [ ]:
import json, os, shutil, sys
from pathlib import Path

# ---- 1. Ask the user to name the notebook -------------------------------
suggested = "dynamic_routh_p2_two_quadrics.ipynb"
typed = input(f"Name for this notebook [{suggested}]: ").strip()
notebook_name = typed or suggested
if not notebook_name.endswith(".ipynb"):
    notebook_name += ".ipynb"
print(f"name chosen: {notebook_name}")

# ---- 2. Ask where to save it, with a pop-up dialog ----------------------
def ask_folder_graphically(initial_name):
    """Open a real Save-As window. Returns a path, or None if impossible."""
    try:
        import tkinter as tk
        from tkinter import filedialog
    except Exception as exc:
        print(f"(no tkinter available: {exc})")
        return None
    if sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
        print("(no graphical display detected: $DISPLAY is not set)")
        return None
    try:
        root = tk.Tk()
        root.withdraw()                 # hide the empty parent window
        root.attributes("-topmost", True)   # put the dialog in front
        chosen = filedialog.asksaveasfilename(
            title="Save this notebook as...",
            initialfile=initial_name,
            defaultextension=".ipynb",
            filetypes=[("Jupyter notebook", "*.ipynb"), ("All files", "*.*")],
        )
        root.destroy()
        return chosen or None
    except Exception as exc:
        print(f"(could not open the dialog: {exc})")
        return None

target = ask_folder_graphically(notebook_name)

if target is None:
    # Fallback for a machine with no display: ask in plain text instead.
    print("Falling back to a typed folder path.")
    default_dir = str(Path.cwd())
    folder = input(f"Folder to save into [{default_dir}]: ").strip() or default_dir
    target = str(Path(folder).expanduser() / notebook_name)

target = Path(target).expanduser()
target.parent.mkdir(parents=True, exist_ok=True)

# ---- 3. Copy this notebook to the chosen place --------------------------
source = Path("notebooks/dynamic_routh_p2_two_quadrics.ipynb")
if not source.is_file():
    # The notebook may have been opened from somewhere else; look nearby.
    hits = list(Path.cwd().rglob(source.name))
    source = hits[0] if hits else None

if source is None:
    print(
        "Could not locate this notebook's own file on disk to copy it.\n"
        f"Use JupyterLab's  File -> Save Notebook As...  and save to:\n    {target}"
    )
else:
    shutil.copyfile(source, target)
    print(f"saved: {target}")